In [6]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split


# ==============================
# Load Data
# ==============================
df = pd.read_csv("ncr_ride_bookings.csv")

# Create datetime features
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df['hour'] = df['datetime'].dt.hour


# ==============================
# Model 1: Simple Model (Hour Only)
# ==============================
print("\nMODEL 1: SIMPLE MODEL")

demand_simple = df.groupby('hour').size().reset_index(name='rides')

X1 = demand_simple[['hour']]
y1 = demand_simple['rides']

model1 = LinearRegression()
model1.fit(X1, y1)

pred1 = model1.predict(X1)

r2_1 = r2_score(y1, pred1)

print("R-squared (Simple Model):", round(r2_1, 3))

# store predictions
demand_simple['predicted_rides'] = pred1

print("\nSample (Simple Model):")
print(demand_simple.head(5))


# ==============================
# Model 2: Improved Model (Hour + Location)
# ==============================
print("\nMODEL 2: IMPROVED MODEL")

demand_advanced = df.groupby(['hour', 'Pickup Location']).size().reset_index(name='rides')

# Encode categorical variable
demand_advanced_encoded = pd.get_dummies(demand_advanced, columns=['Pickup Location'])

X2 = demand_advanced_encoded.drop('rides', axis=1)
y2 = demand_advanced_encoded['rides']

X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
model2.fit(X_train, y_train)

pred2 = model2.predict(X_test)

r2_2 = r2_score(y_test, pred2)

print("R-squared (Improved Model):", round(r2_2, 3))

# show sample predictions
advanced_results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': pred2
})

print("\nSample (Improved Model):")
print(advanced_results.head(5))


# ==============================
# Prepare Predictions for OR
# ==============================
# Predict for full dataset
full_predictions = demand_advanced.copy()
full_predictions['predicted_rides'] = model2.predict(X2)

# Aggregate to hourly demand
hourly_predictions = full_predictions.groupby('hour')['predicted_rides'].sum().reset_index()


# ==============================
# Save Results
# ==============================
os.makedirs("results", exist_ok=True)

# save for OR model
hourly_predictions.to_csv("results/demand_predictions_advanced.csv", index=False)

print("\nPredictions saved successfully.")


# ==============================
# Final Comparison
# ==============================
print("\nMODEL COMPARISON")
print("Simple Model R-squared :", round(r2_1, 3))
print("Improved Model R-squared :", round(r2_2, 3))



MODEL 1: SIMPLE MODEL
R-squared (Simple Model): 0.44

Sample (Simple Model):
   hour  rides  predicted_rides
0     0   1373      2448.830000
1     1   1360      2779.366522
2     2   1339      3109.903043
3     3   1383      3440.439565
4     4   1321      3770.976087

MODEL 2: IMPROVED MODEL


R-squared (Improved Model): 0.918

Sample (Improved Model):
      Actual  Predicted
3907      26  31.017949
2340      26  30.557021
2446      26  30.557021
4127      12  15.683460
1694      45  46.310566

Predictions saved successfully.

MODEL COMPARISON
Simple Model R-squared : 0.44
Improved Model R-squared : 0.918
